# Domain-Adaptive Pretraining (TAPT / MLM) for mmBERT
### Target: Infuse LGBTQIA+ Nuances, Social Media Slang, & Subtle Hate Knowledge
**Backbone Model**: `jhu-clsp/mmbert-base`  
**Pretraining Objective**: Masked Language Modeling (MLM 15%)  
**Corpus**: Filtered LGBTQ+ data from `measuring-hate-speech`, `toxigen-data`, `tides`, and in-domain YouTube context.

In [ ]:
# -----------------------------------------------------------------------------
# 0. CONFIGURATION & HUGGING FACE AUTHENTICATION (OPTIONAL)
# -----------------------------------------------------------------------------
import os
import gc
import glob
import torch
import pandas as pd
import numpy as np

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

BASE_MODEL_NAME = "jhu-clsp/mmbert-base"
OUTPUT_LOCAL_DIR = "./mmbert-queer-hate-adapted"

# Optional: Set your Hugging Face credentials if you want to push to Hub directly
# Set to None if you only want to save locally in Kaggle output
HF_WRITE_TOKEN = None  # e.g. "hf_xxxxxxxxxxxxxxxxxxxx"
HF_REPO_NAME = None    # e.g. "your-username/mmbert-queer-hate-adapted"

EPOCHS = 4
BATCH_SIZE = 32
MAX_LENGTH = 256
LEARNING_RATE = 4e-5
MLM_PROBABILITY = 0.15

if HF_WRITE_TOKEN and HF_WRITE_TOKEN.startswith("hf_"):
    from huggingface_hub import login
    login(token=HF_WRITE_TOKEN)
    print("✅ Logged in to Hugging Face Hub.")
else:
    print("ℹ️ Running in local mode (Model will be saved to Kaggle output).")

In [ ]:
# -----------------------------------------------------------------------------
# 1. LOAD AND FILTER DOMAIN DATASETS
# -----------------------------------------------------------------------------
from datasets import load_dataset, Dataset

collected_texts = []
print("==================== COLLECTING DOMAIN TEXTS ====================")

# --- Dataset A: Measuring Hate Speech (UC Berkeley D-Lab) ---
try:
    print("\n[1/4] Loading 'ucberkeley-dlab/measuring-hate-speech'...")
    mhs_ds = load_dataset("ucberkeley-dlab/measuring-hate-speech", split="train")
    mhs_df = mhs_ds.to_pandas()
    
    cond = (
        (mhs_df.get('target_sexuality', False) == True) |
        (mhs_df.get('target_gender', False) == True) |
        (mhs_df.get('hate_speech_score', 0) > 0.0)
    )
    mhs_filtered = mhs_df[cond]['text'].dropna().astype(str).tolist()
    print(f"  --> Extracted {len(mhs_filtered)} LGBTQ+ & hate speech sentences from MHS.")
    collected_texts.extend(mhs_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading MHS: {e}")

# --- Dataset B: ToxiGen (Subtle / Implicit Toxicity) ---
try:
    print("\n[2/4] Loading 'toxigen/toxigen-data' (annotated split)...")
    tox_ds = load_dataset("toxigen/toxigen-data", name="annotated", split="train")
    tox_df = tox_ds.to_pandas()
    
    if 'target_group' in tox_df.columns:
        mask = tox_df['target_group'].astype(str).str.contains('lgbtq|gay|trans|lesbian|queer', case=False, na=False)
        tox_filtered = tox_df[mask]['text'].dropna().astype(str).tolist()
    else:
        tox_filtered = tox_df['text'].dropna().astype(str).tolist()[:20000]
    print(f"  --> Extracted {len(tox_filtered)} subtle/implicit texts from ToxiGen.")
    collected_texts.extend(tox_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading ToxiGen: {e}")

# --- Dataset C: TIDES (Targeted Identity Disparagement & Exclusion) ---
try:
    print("\n[3/4] Loading 'falseiftrue/tides'...")
    tides_ds = load_dataset("falseiftrue/tides", split="train")
    tides_df = tides_ds.to_pandas()
    
    text_col = 'text' if 'text' in tides_df.columns else tides_df.columns[0]
    tides_filtered = tides_df[text_col].dropna().astype(str).tolist()
    print(f"  --> Extracted {len(tides_filtered)} texts from TIDES.")
    collected_texts.extend(tides_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading TIDES: {e}")

# --- Dataset D: In-Domain YouTube Dataset (StereoQueer) ---
print("\n[4/4] Searching for In-Domain StereoQueer files in Kaggle / Local paths...")
possible_paths = [
    "/kaggle/input/**/train.csv",
    "/kaggle/input/**/test.csv",
    "/kaggle/input/**/dev.csv",
    "data/train.csv",
    "train.csv"
]

in_domain_count = 0
for pattern in possible_paths:
    for filepath in glob.glob(pattern, recursive=True):
        try:
            df_in = pd.read_csv(filepath)
            for col in ['yt_comment', 'yt_title', 'yt_description', 'comment', 'text']:
                if col in df_in.columns:
                    texts = df_in[col].dropna().astype(str).tolist()
                    collected_texts.extend(texts)
                    in_domain_count += len(texts)
            print(f"  --> Added in-domain data from: {filepath}")
        except Exception as err:
            print(f"  ⚠️ Could not read {filepath}: {err}")

# Clean & Deduplicate
cleaned_texts = []
seen = set()
for t in collected_texts:
    clean_t = t.strip()
    if len(clean_t) >= 12 and clean_t not in seen:
        seen.add(clean_t)
        cleaned_texts.append(clean_t)

print(f"\n✨ TOTAL UNIQUE CURATED SENTENCES FOR PRETRAINING: {len(cleaned_texts):,}")

In [ ]:
# -----------------------------------------------------------------------------
# 2. TOKENIZATION & DYNAMIC 15% MASKING COLLATOR
# -----------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling

print(f"Loading base tokenizer & MLM model from: {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME)

raw_dataset = Dataset.from_dict({"text": cleaned_texts})

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True
    )

print("Tokenizing dataset...")
tokenized_dataset = raw_dataset.map(
    tokenize_fn,
    batched=True,
    num_proc=2,
    remove_columns=["text"]
)

# Dynamic Masked Language Modeling Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY
)
print("✅ Tokenization and DataCollator configured.")

In [ ]:
# -----------------------------------------------------------------------------
# 3. TRAINER & PRETRAINING LOOP
# -----------------------------------------------------------------------------
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_LOCAL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("🚀 Starting Domain-Adaptive MLM Pretraining...")
trainer.train()
print("✅ MLM Pretraining Complete!")

In [ ]:
# -----------------------------------------------------------------------------
# 4. SAVE WEIGHTS & PUSH TO HUGGING FACE (OR KAGGLE OUTPUT)
# -----------------------------------------------------------------------------
print(f"Saving adapted model and tokenizer to: {OUTPUT_LOCAL_DIR}...")
model.save_pretrained(OUTPUT_LOCAL_DIR)
tokenizer.save_pretrained(OUTPUT_LOCAL_DIR)

# Push to Hugging Face if configured
if HF_REPO_NAME and HF_WRITE_TOKEN:
    print(f"Pushing to Hugging Face Hub: {HF_REPO_NAME}...")
    model.push_to_hub(HF_REPO_NAME)
    tokenizer.push_to_hub(HF_REPO_NAME)
    print(f"🎉 Model successfully published to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("\n📁 Model saved locally in Kaggle notebook output!")
    print(f"Path: {os.path.abspath(OUTPUT_LOCAL_DIR)}")
    print("You can load it in Task B with:")
    print(f"  config.mmbert_model_name = '{OUTPUT_LOCAL_DIR}'")